# Project Nano-Myo
## Notebook 06 - Repeatability Audit


## Step 1 - Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Nano_Myo'
FEATURE_DIR = PROJECT_ROOT / 'features'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SCHEMA_VERSION = 'e2_only_80feat_wl_ssc_rest25000_v1'
INPUT_DIM = 80
HIDDEN_UNITS = (64, 32)
NUM_CLASSES = 10
REST_CAP = 25000
ACTIVE_SMOTE_TARGET = 5000
VAL_SUBJECTS = ['s8']
SEEDS = [0, 1, 2, 3, 4]
CONFIDENCE = 0.95
LOCKFILE_PATH = RESULTS_DIR / 'requirements.lock.txt'
CSV_PATH = RESULTS_DIR / 'repeatability_results.csv'
SUMMARY_PATH = RESULTS_DIR / 'repeatability_summary.json'

assert SEEDS
print(f'Seeds: {SEEDS}')
print(f'Validation subjects: {VAL_SUBJECTS}')


## Step 2 - Install


In [ ]:
!pip install -q tensorflow==2.20.0 keras==3.14.1 tf-keras==2.20.1 tensorflow-model-optimization==0.8.1 "protobuf>=5.29.1,<6.0.0" imbalanced-learn
!pip freeze > /content/drive/MyDrive/Nano_Myo/results/requirements.lock.txt
print(f'Lockfile saved: {LOCKFILE_PATH}')
print('Restart the runtime before continuing if Colab replaced TensorFlow packages.')


## Step 3 - Imports


In [ ]:
import os
import sys

if 'tensorflow' in sys.modules:
    raise RuntimeError('Restart the runtime before continuing so TensorFlow imports after the environment is configured.')

os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import csv
import json
import platform
import random
import subprocess
from collections import Counter
from datetime import datetime, timezone

import imblearn
import keras
import numpy as np
import scipy
import sklearn
import tf_keras
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from imblearn.over_sampling import SMOTE
from scipy import stats

try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISTIC_OPS_ENABLED = True
except Exception as exc:
    DETERMINISTIC_OPS_ENABLED = False
    print(f'Deterministic op mode unavailable: {exc}')


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def git_commit():
    for root in [PROJECT_ROOT, Path.cwd()]:
        try:
            result = subprocess.run(
                ['git', '-C', str(root), 'rev-parse', 'HEAD'],
                capture_output=True,
                text=True,
                timeout=10,
            )
            if result.returncode == 0 and result.stdout.strip():
                return result.stdout.strip()
        except Exception:
            pass
    return 'unavailable'


ENVIRONMENT = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
    'imbalanced_learn': imblearn.__version__,
    'keras': keras.__version__,
    'tf_keras': tf_keras.__version__,
    'tensorflow': tf.__version__,
    'tf_model_optimization': tfmot.__version__,
    'deterministic_ops_enabled': DETERMINISTIC_OPS_ENABLED,
    'gpu_devices': [device.name for device in tf.config.list_physical_devices('GPU')],
    'git_commit': git_commit(),
}

print(json.dumps(ENVIRONMENT, indent=2))


## Step 4 - Load Artifacts


In [ ]:
paths = {
    'X_train': FEATURE_DIR / f'X_train_{FEATURE_SCHEMA_VERSION}.npy',
    'y_train': FEATURE_DIR / f'y_train_{FEATURE_SCHEMA_VERSION}.npy',
    'subjects_train': FEATURE_DIR / f'subjects_train_{FEATURE_SCHEMA_VERSION}.npy',
    'X_test': FEATURE_DIR / f'X_test_{FEATURE_SCHEMA_VERSION}.npy',
    'y_test': FEATURE_DIR / f'y_test_{FEATURE_SCHEMA_VERSION}.npy',
    'subjects_test': FEATURE_DIR / f'subjects_test_{FEATURE_SCHEMA_VERSION}.npy',
    'metadata': FEATURE_DIR / f'nano_myo_features_holdout_s9_s10_{FEATURE_SCHEMA_VERSION}_metadata.json',
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

with open(paths['metadata'], 'r', encoding='utf-8') as f:
    metadata = json.load(f)

if metadata['feature_schema_version'] != FEATURE_SCHEMA_VERSION:
    raise ValueError(metadata['feature_schema_version'])
if metadata['exercise_used'] != 'E2':
    raise ValueError(metadata['exercise_used'])
if metadata['input_dim'] != INPUT_DIM:
    raise ValueError(metadata['input_dim'])
if set(metadata['feature_groups']) != {'mav', 'zc', 'rms', 'wl', 'ssc'}:
    raise ValueError(metadata['feature_groups'])

X_train_all = np.load(paths['X_train']).astype(np.float32)
y_train_all = np.load(paths['y_train']).astype(np.int64)
subjects_train_all = np.load(paths['subjects_train'], allow_pickle=True)
X_test = np.load(paths['X_test']).astype(np.float32)
y_test = np.load(paths['y_test']).astype(np.int64)
subjects_test = np.load(paths['subjects_test'], allow_pickle=True)
TARGET_LABELS = {int(key): value for key, value in metadata['target_labels'].items()}

if X_train_all.shape[1] != INPUT_DIM or X_test.shape[1] != INPUT_DIM:
    raise ValueError((X_train_all.shape, X_test.shape))
if not np.any(np.isin(subjects_train_all, VAL_SUBJECTS)):
    raise ValueError(VAL_SUBJECTS)

print(f'X_train_all: {X_train_all.shape}')
print(f'X_test: {X_test.shape}')
print(f'Train subjects: {sorted(np.unique(subjects_train_all).tolist())}')
print(f'Test subjects: {sorted(np.unique(subjects_test).tolist())}')
print(f'Test counts: {dict(Counter(y_test.tolist()))}')


## Step 5 - Metrics


In [ ]:
COST_MATRIX = np.array([
    [0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    [0.8, 0.0, 1.0, 0.4, 0.4, 0.6, 0.6, 0.3, 0.3, 0.4],
    [0.8, 1.0, 0.0, 0.3, 0.3, 0.6, 0.6, 0.3, 0.3, 0.3],
    [0.8, 0.4, 0.3, 0.0, 0.2, 0.5, 0.5, 0.3, 0.3, 0.2],
    [0.8, 0.4, 0.3, 0.2, 0.0, 0.5, 0.5, 0.3, 0.3, 0.2],
    [0.8, 0.5, 0.5, 0.4, 0.4, 0.0, 1.0, 0.4, 0.4, 0.4],
    [0.8, 0.5, 0.5, 0.4, 0.4, 1.0, 0.0, 0.4, 0.4, 0.4],
    [0.8, 0.3, 0.3, 0.3, 0.3, 0.4, 0.4, 0.0, 0.6, 0.3],
    [0.8, 0.3, 0.3, 0.3, 0.3, 0.4, 0.4, 0.6, 0.0, 0.3],
    [0.8, 0.4, 0.3, 0.2, 0.2, 0.4, 0.4, 0.3, 0.3, 0.0],
], dtype=np.float32)


def standard_accuracy(y_true, y_pred):
    return float(np.mean(y_pred == y_true))


def functional_metrics(y_true, y_pred):
    costs = COST_MATRIX[y_true, y_pred]
    return float(1.0 - np.mean(costs)), float(np.mean(costs == 1.0))


## Step 6 - Helpers


In [ ]:
def build_model(seed):
    inputs = tf.keras.Input(shape=(INPUT_DIM,))
    x = tf.keras.layers.Dense(
        HIDDEN_UNITS[0],
        activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 1),
    )(inputs)
    x = tf.keras.layers.Dense(
        HIDDEN_UNITS[1],
        activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 2),
    )(x)
    outputs = tf.keras.layers.Dense(
        NUM_CLASSES,
        activation='softmax',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed + 3),
    )(x)
    return tf.keras.Model(inputs, outputs)


def balance_training_partition(X, y, seed):
    rest_idx = np.flatnonzero(y == 0)
    active_idx = np.flatnonzero(y != 0)
    if rest_idx.shape[0] > REST_CAP:
        rng = np.random.default_rng(seed)
        rest_idx = rng.choice(rest_idx, size=REST_CAP, replace=False)
    keep = np.sort(np.concatenate([active_idx, rest_idx]))
    X_balanced = X[keep]
    y_balanced = y[keep]
    counts = Counter(y_balanced.tolist())
    strategy = {
        class_id: ACTIVE_SMOTE_TARGET
        for class_id in range(1, NUM_CLASSES)
        if counts.get(class_id, 0) < ACTIVE_SMOTE_TARGET and counts.get(class_id, 0) > 5
    }
    if strategy:
        smote = SMOTE(
            sampling_strategy=strategy,
            random_state=seed,
            k_neighbors=5,
        )
        X_balanced, y_balanced = smote.fit_resample(X_balanced, y_balanced)
    return X_balanced.astype(np.float32), y_balanced.astype(np.int64)


def convert_int8(qat_model, X_repr, seed):
    rng = np.random.default_rng(seed)
    idx = rng.choice(X_repr.shape[0], size=min(1000, X_repr.shape[0]), replace=False)
    sample = X_repr[idx].astype(np.float32)

    def representative_dataset():
        for row in sample:
            yield [row.reshape(1, INPUT_DIM).astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    return converter.convert()


def run_tflite(tflite_bytes, X):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_scale, input_zero_point = input_details['quantization']
    output_scale, output_zero_point = output_details['quantization']
    predictions = []

    for row in X.astype(np.float32):
        sample = row.reshape(1, INPUT_DIM)
        if input_details['dtype'] in (np.int8, np.uint8):
            sample = np.round(sample / input_scale + input_zero_point)
            sample = np.clip(
                sample,
                np.iinfo(input_details['dtype']).min,
                np.iinfo(input_details['dtype']).max,
            )
            sample = sample.astype(input_details['dtype'])
        else:
            sample = sample.astype(input_details['dtype'])

        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        if output_details['dtype'] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale
        predictions.append(int(np.argmax(output, axis=1)[0]))

    return np.asarray(predictions, dtype=np.int64)


## Step 7 - Run Seed


In [ ]:
def run_once(seed):
    tf.keras.backend.clear_session()
    set_all_seeds(seed)

    val_mask = np.isin(subjects_train_all, VAL_SUBJECTS)
    fit_mask = ~val_mask
    X_fit_raw = X_train_all[fit_mask]
    y_fit_raw = y_train_all[fit_mask]
    X_val = X_train_all[val_mask]
    y_val = y_train_all[val_mask]

    if X_fit_raw.shape[0] == 0 or X_val.shape[0] == 0:
        raise ValueError('Empty fit or validation partition')

    X_fit, y_fit = balance_training_partition(X_fit_raw, y_fit_raw, seed)

    float_model = build_model(seed)
    float_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    float_model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
            )
        ],
        verbose=0,
    )

    qbase = build_model(seed)
    qbase.set_weights(float_model.get_weights())
    qat_model = tfmot.quantization.keras.quantize_model(qbase)
    qat_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    qat_model.fit(
        X_fit,
        y_fit,
        validation_data=(X_val, y_val),
        epochs=15,
        batch_size=64,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True,
            )
        ],
        verbose=0,
    )

    qat_pred = np.argmax(qat_model.predict(X_test, batch_size=256, verbose=0), axis=1)
    tflite_bytes = convert_int8(qat_model, X_fit, seed)
    y_pred = run_tflite(tflite_bytes, X_test)
    functional_accuracy, dangerous_confusion_rate = functional_metrics(y_test, y_pred)

    result = {
        'seed': int(seed),
        'tflite_bytes': int(len(tflite_bytes)),
        'standard_accuracy': standard_accuracy(y_test, y_pred),
        'functional_accuracy': functional_accuracy,
        'dangerous_confusion_rate': dangerous_confusion_rate,
        'qat_tflite_match_rate': float(np.mean(y_pred == qat_pred)),
    }

    for subject in sorted(np.unique(subjects_test).tolist()):
        mask = subjects_test == subject
        subject_functional_accuracy, subject_dangerous_confusion_rate = functional_metrics(y_test[mask], y_pred[mask])
        result[f'{subject}_standard_accuracy'] = standard_accuracy(y_test[mask], y_pred[mask])
        result[f'{subject}_functional_accuracy'] = subject_functional_accuracy
        result[f'{subject}_dangerous_confusion_rate'] = subject_dangerous_confusion_rate

    return result


## Step 8 - Run Repeats


In [ ]:
records = []

for seed in SEEDS:
    record = run_once(seed)
    records.append(record)
    print(
        f"seed {seed:>2} | size {record['tflite_bytes']:>6} B | "
        f"std {record['standard_accuracy']:.4f} | "
        f"func {record['functional_accuracy']:.4f} | "
        f"danger {record['dangerous_confusion_rate']:.4f} | "
        f"match {record['qat_tflite_match_rate']:.4f}"
    )


## Step 9 - Summarize


In [ ]:
def summarize(values, confidence=CONFIDENCE):
    arr = np.asarray(values, dtype=np.float64)
    n = arr.size
    mean = float(arr.mean())
    sd = float(arr.std(ddof=1)) if n > 1 else 0.0
    if n > 1:
        half_width = float(stats.t.ppf(0.5 + confidence / 2, df=n - 1) * sd / np.sqrt(n))
    else:
        half_width = 0.0
    return {
        'mean': mean,
        'sd': sd,
        'ci_low': mean - half_width,
        'ci_high': mean + half_width,
        'n': int(n),
    }


metric_names = [
    'standard_accuracy',
    'functional_accuracy',
    'dangerous_confusion_rate',
    'tflite_bytes',
    'qat_tflite_match_rate',
]

summary = {
    metric_name: summarize([record[metric_name] for record in records])
    for metric_name in metric_names
}

header = '{:<32}{:>10}{:>10}{:>24}'.format('metric', 'mean', 'sd', '95% CI')
print(header)
print('-' * len(header))

for metric_name in metric_names:
    values = summary[metric_name]
    if metric_name == 'tflite_bytes':
        ci = '[{:.1f}, {:.1f}]'.format(values['ci_low'], values['ci_high'])
        print('{:<32}{:>10.1f}{:>10.1f}{:>24}'.format(metric_name, values['mean'], values['sd'], ci))
    else:
        ci = '[{:.4f}, {:.4f}]'.format(values['ci_low'], values['ci_high'])
        print('{:<32}{:>10.4f}{:>10.4f}{:>24}'.format(metric_name, values['mean'], values['sd'], ci))


## Step 10 - Save


In [ ]:
fields = list(records[0].keys())

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(records)

config = {
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'input_dim': INPUT_DIM,
    'hidden_units': list(HIDDEN_UNITS),
    'num_classes': NUM_CLASSES,
    'rest_cap': REST_CAP,
    'active_smote_target': ACTIVE_SMOTE_TARGET,
    'val_subjects': VAL_SUBJECTS,
    'test_subjects': sorted(np.unique(subjects_test).tolist()),
    'seeds': SEEDS,
    'confidence': CONFIDENCE,
    'protocol_note': 'Repeatability audit uses subject-grouped validation before SMOTE.',
    'normalization_note': 'Notebook 02 artifacts use per-subject standardization fitted over each subject session.',
}

summary_document = {
    'environment': ENVIRONMENT,
    'config': config,
    'per_seed': records,
    'summary': summary,
}

SUMMARY_PATH.write_text(json.dumps(summary_document, indent=2), encoding='utf-8')

print(f'Saved: {CSV_PATH}')
print(f'Saved: {SUMMARY_PATH}')
print(f'Saved: {LOCKFILE_PATH}')
